In [ ]:
# Task 2 (Riley) - Data Acquisition and Inspection

import pandas as pd 
import numpy as np

print(" --- DATA AQUISITION ---\n")

# Loading the crash dataset (second version)
df = pd.read_csv("ardd_fatalities.csv", low_memory=False)

# Shows the number of records (rows) and features (columns) in the dataset
print("Number of records:", df.shape[0])
print("Number of features/columns:", df.shape[1])
print()

# 2) Inspect missing values and data types
print("--- MISSING VALUES & DATA TYPES ---")
missing_info = pd.DataFrame({
    'Data Type': df.dtypes,
    'Missing Count': df.isnull().sum(),
    'Missing Percentage': (df.isnull().mean() * 100).round(2)
}) 
print(missing_info)
print()

# 3) Inspect value ranges (numerical features)
# Suppress scientific notation and display floats rounded to 2 decimal places
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("--- SENTINEL VALUE INSPECTION (-9) ---")
sentinel_summary = []
for col in df.columns:
    # Match both numerical -9 and text '-9' / '-9.0'
    count = (df[col] == -9).sum() + (df[col].astype(str).str.strip().isin(['-9', '-9.0'])).sum()
    if count > 0:
        pct = (count / len(df)) * 100
        sentinel_summary.append({'Column': col, '-9 Count': count, 'Percentage': f"{pct:.2f}%"})

sentinel_df = pd.DataFrame(sentinel_summary)
if not sentinel_df.empty:
    print("Columns containing '-9' sentinel placeholders:")
    print(sentinel_df.to_string(index=False))
else:
    print("No '-9' sentinel values found.")
print()

print("\n--- NUMERICAL VALUE RANGES ---")
print(df.describe().T[['min', 'mean', '50%', 'max']])
print()

# 4) Check for duplicate records
print("\n--- DUPLICATE RECORDS ---")
duplicates = df.duplicated().sum()
print(f"\nTotal Duplicate Records: {duplicates} ({(duplicates / len(df) * 100):.2f}%)")
print()

print("--- CATEGORICAL INCONSISTENCIES ---")
categorical_cols = df.select_dtypes(include=['object', 'string']).columns

for col in categorical_cols:
    non_nulls = df[col].dropna().astype(str)
    
    # Check leading/trailing whitespace
    whitespace_count = non_nulls.str.contains(r'^\s+|\s+$', regex=True).sum()
    
    # Check for mixed casing (e.g., both 'Bus' and 'BUS' present)
    unique_raw = set(non_nulls.unique())
    unique_upper = set(non_nulls.str.upper().str.strip().unique())
    casing_issues = len(unique_raw) - len(unique_upper)
    
    if whitespace_count > 0 or casing_issues > 0:
        print(f"Column '{col}':")
        if whitespace_count > 0:
            print(f"  - Whitespace issues: {whitespace_count} entries containing leading/trailing spaces.")
        if casing_issues > 0:
            print(f"  - Mixed casing variants detected ({casing_issues} overlapping case duplicates).")
print()

# 5) Check for unique values and potential inconsistencies for categorical features
print("\n--- UNIQUE VALUES & CATEGORICAL SUMMARY ---")
categorical_cols = df.select_dtypes(include=['object', 'string']).columns
for col in categorical_cols:
    unique_vals = df[col].unique()
    print(f"\nColumn '{col}' - {len(unique_vals)} unique values:")
    print(df[col].value_counts(dropna=False).head(10))  # Top 10 categories
    print()

print("--- DOMAIN LOGIC & HIGH VALUE VERIFICATION ---")

# Age Verification
if 'Age' in df.columns:
    age_num = pd.to_numeric(df['Age'], errors='coerce')
    invalid_age = df[(age_num < 0) & (~df['Age'].astype(str).str.contains('-9'))]
    high_age = df[age_num >= 90]
    print(f"Age - Max Value Detected: {age_num.max()}")
    print(f"Age - Records with Age >= 90 (Valid High Values): {len(high_age)}")
    print(f"Age - Invalid Values (excluding -9 sentinels): {len(invalid_age)}")

# Speed Limit Verification
if 'Speed Limit' in df.columns:
    speed_num = pd.to_numeric(df['Speed Limit'], errors='coerce')
    valid_high_speed = df[(speed_num >= 100) & (speed_num <= 110)]
    invalid_speed = df[(speed_num > 130) | ((speed_num < 0) & (~df['Speed Limit'].astype(str).str.contains('-9')))]
    print(f"Speed Limit - Max Speed Detected: {speed_num.max()} km/h")
    print(f"Speed Limit - Highways/Expressways (100-110 km/h): {len(valid_high_speed)} records retained")
    print(f"Speed Limit - Invalid Values (>130 or <0, excluding -9): {len(invalid_speed)}")
print()

# 6) Check for specific domain logic & inconsistencies 
print("\n--- SPECIFIC INCONSISTENCIES ---")
# A. Age Domain Check
if 'Age' in df.columns:
    invalid_age = df[(df['Age'] < 0) | (df['Age'] > 115)]
    print(f"Invalid/Placeholder Age Records (<0 or >115): {len(invalid_age)}")

# B. Speed Limit Domain & Data Type Check
if 'Speed Limit' in df.columns:
    # Convert safely to numeric (non-numeric text strings like 'Unknown' become NaN)
    speed_numeric = pd.to_numeric(df['Speed Limit'], errors='coerce')
    
    # Check for invalid numerical speed limits outside 0 to 130 km/h
    invalid_speed = df[(speed_numeric < 0) | (speed_numeric > 130)]
    print(f"Invalid Speed Limit Records (<0 or >130 km/h): {len(invalid_speed)}")
    
    # Check for non-numeric/text values in the speed column
    non_numeric_speed = df[speed_numeric.isna() & df['Speed Limit'].notna()]
    print(f"Non-numeric Speed Limit Entries (e.g. 'Unknown'): {len(non_numeric_speed)}")

# C. Check for string whitespace
print("\nChecking for string whitespace inconsistencies...")
for col in df.columns:
    has_whitespace = df[col].dropna().astype(str).str.contains(r'^\s+|\s+$', regex=True).any()
    if has_whitespace:
        print(f" - Column '{col}' contains entries with leading or trailing whitespace.")

 --- DATA AQUISITION ---

Number of records: 55360
Number of features/columns: 24

--- MISSING VALUES & DATA TYPES ---
                              Data Type  Missing Count  Missing Percentage
Crash ID                          int64              0                0.00
State                               str              0                0.00
Month                             int64              0                0.00
Year                              int64              0                0.00
Dayweek                             str              0                0.00
Time                                str             53                0.10
Crash Type                          str              5                0.01
Bus Involvement                     str              4                0.01
Heavy Rigid Truck Involvement       str              4                0.01
Articulated Truck Involvement       str              4                0.01
Speed Limit                         str              0  

In [ ]:
# Task 2 (Riley) - Data Acquisition, Inspection, and Documentation

import pandas as pd 
import numpy as np

# Adjust pandas display options for clean terminal output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("--- DATA ACQUISITION ---")

# 1) Load the Australian Road Deaths Database
df = pd.read_csv("ardd_fatalities.csv", low_memory=False)

# Check for duplicates on Crash ID
duplicates = df[df.duplicated(subset=['Crash ID'], keep=False)]
print(f"Found {len(duplicates)} total rows sharing a non-unique 'Crash ID'.")

# Show top 5 duplicate records for key columns
print("\nSample Duplicate Records:")
print(duplicates[['Crash ID', 'State', 'Year', 'Gender', 'Age']].head(5))

# Check whitespace across columns (Fixed Line 87)
print("\n--- WHITESPACE CHECK ---")
for col in df.columns:
    has_ws = df[col].dropna().astype(str).str.contains(r'^\s+|\s+$', regex=True).any()
    if has_ws:
        print(f"Column '{col}' contains leading/trailing whitespace.")

# Display dataset dimensions
print(f"Number of records (rows): {df.shape[0]:,}")
print(f"Number of features (columns): {df.shape[1]}")
print()

# 2) Inspect missing values and data types
print("--- MISSING VALUES & DATA TYPES ---")
missing_info = pd.DataFrame({
    'Data Type': df.dtypes.astype(str),
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().mean() * 100).round(2)
}) 
print(missing_info)
print()

# 3) Inspect numerical value ranges
print("--- NUMERICAL VALUE RANGES ---")
numeric_summary = df.describe().T[['min', 'mean', '50%', 'max']]
print(numeric_summary)
print()

# 4) Check for duplicate records across all columns
print("--- DUPLICATE RECORDS ---")
duplicates = df.duplicated().sum()
duplicate_pct = (duplicates / len(df)) * 100
print(f"Total Duplicate Records: {duplicates:,} ({duplicate_pct:.2f}%)")
print()

# 5) Check unique values and value distributions for categorical features
print("--- UNIQUE VALUES & CATEGORICAL SUMMARY ---")
categorical_cols = df.select_dtypes(include=['object', 'string']).columns

for col in categorical_cols:
    unique_count = df[col].nunique(dropna=False)
    print(f"Column '{col}' - {unique_count} unique values:")
    print(df[col].value_counts(dropna=False).head(10))
    print("-" * 50)
print()

# 6) Check specific domain logic & data anomalies
print("--- SPECIFIC INCONSISTENCIES ---")

# A. Age Range Check
if 'Age' in df.columns:
    invalid_age = df[(df['Age'] < 0) | (df['Age'] > 115)]
    print(f"Invalid/Placeholder Age Records (<0 or >115 yrs): {len(invalid_age)}")

# B. Speed Limit Range & Type Check
if 'Speed Limit' in df.columns:
    speed_numeric = pd.to_numeric(df['Speed Limit'], errors='coerce')
    
    invalid_speed = df[(speed_numeric < 0) | (speed_numeric > 130)]
    print(f"Invalid Speed Limit Values (<0 or >130 km/h): {len(invalid_speed)}")
    
    non_numeric_speed = df[speed_numeric.isna() & df['Speed Limit'].notna()]
    print(f"Non-numeric Speed Limit Entries (e.g., 'Unknown'): {len(non_numeric_speed)}")

# C. Whitespace Inconsistency Check
print("\nChecking for string whitespace inconsistencies...")
whitespace_cols = []
for col in df.select_dtypes(include=['object', 'string']).columns:
    has_whitespace = df[col].dropna().astype(str).str.contains(r'^\s+|\s+$', regex=True).any()
    if has_whitespace:
        whitespace_cols.append(col)

if whitespace_cols:
    for col in whitespace_cols:
        print(f" - Column '{col}' contains leading or trailing whitespace.")
else:
    print(" - No leading/trailing whitespace detected.")


<>:93: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:93: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/var/folders/2s/kthn789170x693nqhn_9kw100000gn/T/ipykernel_38495/3532589821.py:93: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  has_whitespace = df[col].dropna().astype(str).str.contains(r'^\s+|\s+$', regex=True).any()


--- DATA ACQUISITION ---
Found 9717 total rows sharing a non-unique 'Crash ID'.

Sample Duplicate Records:
    Crash ID State  Year Gender  Age
3   20233149   Qld  2023   Male   37
6   20235077    WA  2023   Male   29
8   20233149   Qld  2023   Male   39
12  20237018    NT  2023   Male   -9
14  20235097    WA  2023   Male   60

--- WHITESPACE CHECK ---
Column 'Gender' contains leading/trailing whitespace.


'\n# Display dataset dimensions\nprint(f"Number of records (rows): {df.shape[0]:,}")\nprint(f"Number of features (columns): {df.shape[1]}")\nprint()\n\n# 2) Inspect missing values and data types\nprint("--- MISSING VALUES & DATA TYPES ---")\nmissing_info = pd.DataFrame({\n    \'Data Type\': df.dtypes.astype(str),\n    \'Missing Count\': df.isnull().sum(),\n    \'Missing Percentage (%)\': (df.isnull().mean() * 100).round(2)\n}) \nprint(missing_info)\nprint()\n\n# 3) Inspect numerical value ranges\nprint("--- NUMERICAL VALUE RANGES ---")\nnumeric_summary = df.describe().T[[\'min\', \'mean\', \'50%\', \'max\']]\nprint(numeric_summary)\nprint()\n\n# 4) Check for duplicate records across all columns\nprint("--- DUPLICATE RECORDS ---")\nduplicates = df.duplicated().sum()\nduplicate_pct = (duplicates / len(df)) * 100\nprint(f"Total Duplicate Records: {duplicates:,} ({duplicate_pct:.2f}%)")\nprint()\n\n# 5) Check unique values and value distributions for categorical features\nprint("--- UNIQUE